In [1]:
import findspark # type: ignore
findspark.init()
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName('Local Batch') \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .config("spark.mongodb.output.uri", "mongodb://127.0.0.1:27017/crypto_database.hourly_stats") \
    .getOrCreate()

25/12/29 11:29:54 WARN Utils: Your hostname, node1 resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
25/12/29 11:29:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/spark-3.1.2-bin-hadoop2.7/jars/ivy-2.4.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vagrant/.ivy2/cache
The jars for the packages stored in: /home/vagrant/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eb1df27b-d899-43a3-a254-6b429a057728;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 in central
	found org.mongodb#mongodb-driver-sync;4.0.5 in central
	found org.mongodb#bson;4.0.5 in central
	found org.mongodb#mongodb-driver-core;4.0.5 in central
downloading https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.12/3.0.1/mongo-spark-connector_2.12-3.0.1.jar ...
	[SUCCESSFUL ] org.mongodb.spark#mongo-spark-connector_2.12;3.0.1!mongo-spark-connector_2.12.jar (135ms)
downloading https://repo1.maven.org/maven2/org/mongodb/mongodb-driver-sync/4.0.5/mongodb-driver-sync-4.0.5.jar ...
	[SUCCESSFUL ] org.mongodb#mongodb-driver-sync;4.0.5!mongodb-driver-sync.jar (49ms)
downloading https://repo1.maven.o

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, hour, stddev, mean
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# target_time = datetime.now()
# target_year = target_time.year
# target_month = target_time.month
# target_day = target_time.day
# target_hour = target_time.hour

target_year = 2025
target_month = 12
target_day = 16
target_hour = 1

path = 'file:///home/vagrant/prices.parquet'

In [4]:
df = spark.read.parquet(path) \
    .filter((col("year") == target_year) &
            (col("month") == target_month) &
            (col("day") == target_day) &
            (hour(col("timestamp")) == target_hour))

window = Window.partitionBy("currency","year", "month", "day", "hour").orderBy("timestamp")

df_enriched = df.withColumn("hour", hour(col("timestamp"))) \
                .withColumn("open_price", F.first("price").over(window)) \
                .withColumn("close_price", F.last("price").over(window))

# 1. Najprostsze statystystyki godzinowe

final_batch_view = df_enriched.groupBy("currency", "year", "month", "day", "hour") \
                              .agg(F.first("open_price").alias("open"),
                                   F.max("price").alias("high"),
                                   F.min("price").alias("low"),
                                   F.last("close_price").alias("close"),
                                   F.count("price").alias("read_count"),
                                   mean(col("price")).alias("mean_price"), 
                                   stddev(col("price")).alias("std_price"))

final_batch_view = final_batch_view.withColumn("return_pct", (col("close") - col("open")) / col("open") * 100) \
                                   .withColumn("spread_pct", (col("high") - col("low")) / col("low") * 100)

In [5]:
final_batch_view.show()

+--------+----+-----+---+----+-------------------+-------------------+-------------------+------------------+----------+-------------------+--------------------+--------------------+------------------+
|currency|year|month|day|hour|               open|               high|                low|             close|read_count|         mean_price|           std_price|          return_pct|        spread_pct|
+--------+----+-----+---+----+-------------------+-------------------+-------------------+------------------+----------+-------------------+--------------------+--------------------+------------------+
|ethereum|2025|   12| 16|   1| 2459.9432126576858|    2782.0385035771| 2188.5397768693765|  2534.29717936578|       720| 2496.2225838063914|    95.3417154677767|   3.022588746175288| 27.11848022962146|
|dogecoin|2025|   12| 16|   1|0.19834677682311946|0.21443672642056633|0.18485059428403108|0.1992222948673822|       720|0.19997203538345443|0.005111847872934501|  0.4414077497430112|16.0054298

In [6]:
final_batch_view.write \
    .format("mongo") \
    .mode("append") \
    .save()